In [ ]:
!pip install pyhealth

In [ ]:
from pyhealth.datasets import MIMIC3Dataset
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# making sure it exists
!ls "/content/drive/MyDrive"

In [ ]:
!ls "/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3"

In [ ]:
# importing the database and pointing the root to the correct directory path
mimic_dataset = MIMIC3Dataset(
    root="/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    dev=False
)

mimic_dataset.stats()

In [ ]:
# Necessary Imports

from pyhealth.tasks import ReadmissionPredictionMIMIC3
import datetime as dt
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RNN
from pyhealth.trainer import Trainer

In [ ]:
# Configure readmission horizon to be 30 days
W = dt.timedelta(days=30)

from pyhealth.utils import set_seed

# Configure seed to 42 for reproducability
set_seed(42)


# Utilizing the ReadmissionPrediction task 
readmission_task = ReadmissionPredictionMIMIC3(window=W, exclude_minors=False)
sample_data = mimic_dataset.set_task(task=readmission_task)

# STEP 3: Split and create dataloaders
train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_data, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# STEP 4: Define model
model = RNN(
  dataset=sample_data,
)

# STEP 5: Train
trainer = Trainer(model=model)
trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc",
)

# STEP 6: Evaluate
trainer.evaluate(test_dataloader)